# L5b: Multiple Asset Geometric Brownian Motion
In the last few lectures we've modeled the share price of a single firm using binomial (or multinomial) lattices and geometric Brownian motion (GBM). The question that we explore today is how do we extend these ideas to several firms whose prices may move together? 

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
>
> * **Update and extend GBM:** Update single asset GBM parameters with an exponential moving average. Write the multiple asset GBM model, explain how a loading matrix introduces correlated fluctuations, and use the exact one-step transition to simulate prices.
> * **Estimate and interpret covariance:** Calculate the covariance matrix from historical growth rates, interpret its entries, and convert it to the covariance rate used by the price model.
> * **Compare portfolio allocations:** Relate initial weights to buy-and-hold wealth, generate long-only allocations with the Dirichlet distribution, and compare them using the mean and variance of the linear growth-rate proxy.

Firms respond to common market news and economic conditions, so we need to describe the relationships between their price movements. In this lecture, we use a covariance matrix to represent those relationships and extend the single asset model to correlated assets.

Let's get started!

___

## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Update GBM parameters with an exponential moving average](../L5a/CHEME-5660-L5a-Example-EMA-SAGBM-Fall-2026.ipynb). Carried over from L5a for the concept review. Can giving recent observations more weight improve the forecasts from a GBM model? We initialize mean growth and volatility from the 2014–2024 estimates, update them as 2025 prices arrive, and compare their forecasts with the frozen model. We measure whether updating volatility alone or both parameters improves the coverage and width of the forecasts' 95% growth-rate prediction bands.

> [▶ Compute the covariance matrix for our dataset](CHEME-5660-L5b-Example-CovarianceMatrix-Fall-2026.ipynb). How do we measure whether firms' growth rates move together? We compute the covariance matrix from historical growth rates, convert it to the covariance rate used by the multiple asset model, and verify the corresponding volatilities against our L4b estimates. We then examine a pair of firms to interpret their covariance and correlation.

> [▶ Sample portfolio weights with the Dirichlet distribution](CHEME-5660-L5b-Example-Dirichlet-PortfolioWeights-Fall-2026.ipynb). How does dividing our investment among firms affect portfolio growth and risk? We sample long-only allocations using the Dirichlet distribution, examine how its concentration parameters shape the weights, and compare the estimated growth and risk of the sampled portfolios. We also track buy-and-hold wealth to see how the fractions invested in each firm change as prices move.

Optional examples on covariance estimation and changing correlations are listed in the Optional Advanced Material section at the end of this lecture.

___


## Concept Review: Updating GBM Parameters with an Exponential Moving Average
Assuming fixed parameters in the geometric Brownian motion (GBM) model is a common criticism. An __exponential moving average (EMA)__ lets us update the mean growth and volatility estimates as new observations arrive, giving recent observations more weight.

> __Proposition: Exponentially Weighted GBM Estimates__
>
> Let $g_k=\ln(S_k/S_{k-1})/\Delta t$ be the observed growth rate ($\mathrm{year}^{-1}$), with positive price observations spaced $\Delta t>0$ years apart. At entry index $s$, initialize its mean and variance from the training estimates: $m_s=\hat\mu_{g,0}$ and $v_s=\hat\sigma_0^2/\Delta t$.
>
> For a decay factor $0<\lambda<1$ and each new observation $k>s$, the weighted moments and corresponding GBM estimates satisfy:
>
> $$
> \begin{aligned}
> \delta_k&=g_k-m_{k-1}, &&\text{(innovation)}\\
> m_k&=m_{k-1}+(1-\lambda)\delta_k, &&\text{(mean update)}\\
> v_k&=\lambda\left[v_{k-1}+(1-\lambda)\delta_k^2\right], &&\text{(variance update)}\\[4pt]
> \hat\mu_{g,k}&=m_k,\qquad
> \hat\sigma_k=\sqrt{v_k\Delta t},\qquad
> \hat\mu_k=\hat\mu_{g,k}+\frac12\hat\sigma_k^2. &&\text{(GBM parameters)}
> \end{aligned}
> $$
>
> Here, $m_k$ and $v_k$ are the weighted mean ($\mathrm{year}^{-1}$) and variance ($\mathrm{year}^{-2}$) of growth rates; $\hat\mu_{g,k}$, $\hat\sigma_k$, and $\hat\mu_k$ are mean growth, volatility, and drift. The variance accounts for the changing mean and has no finite-sample unbiased correction.
>
> __Where is this coming from?__
>
> [▶ Derivation of the EMA parameter updates](../L5a/advanced/ema-derivation/CHEME-5660-L5a-Derivation-EMA-SAGBM-Fall-2026.ipynb). Why does the variance update contain an extra factor of $\lambda$? We derive the exponential weights, centered variance, and conversion to GBM parameters.

__Half-life__: We choose $\lambda=2^{-1/H_{\mathrm{half}}}$, where $H_{\mathrm{half}}$ is the number of observations over which an older weight halves. The default 21-observation half-life gives $\lambda\approx0.9675$. A shorter half-life responds faster to recent changes but also follows more of their noise.

> __Example:__
>
> [▶ Update GBM parameters with an exponential moving average](../L5a/CHEME-5660-L5a-Example-EMA-SAGBM-Fall-2026.ipynb). We compare frozen parameters, updated volatility, and updated mean growth with volatility using the same observations, sale dates, and NPV targets. We evaluate the forecasts using the coverage and width of their 95% growth-rate prediction bands.

With the single asset model now able to update its parameters, we turn to several firms whose prices move together.

___

## Company Profile: Bridgewater Associates

[Bridgewater Associates](https://www.bridgewater.com/) is a global macro investment firm that manages portfolios for pension funds, endowments, and sovereign wealth funds. [Ray Dalio](https://www.bridgewater.com/our-founder) founded the firm in 1975. Its process is systematic: the firm writes down cause-and-effect rules for how economies and markets work, tests them against historical data, and builds portfolios around how assets move together.

> __What makes Bridgewater distinctive?__
>
> * __All Weather:__ Launched in 1996, [All Weather](https://www.bridgewater.com/research-and-insights/the-all-weather-story) is designed to hold up whether growth and inflation come in above or below expectations. Because stocks, nominal bonds, inflation-linked bonds, and commodities respond differently to the same news, the portfolio balances risk across them rather than dollars. Dalio's article [Engineering Targeted Returns and Risks](https://bridgewater.brightspotcdn.com/fa/e3/d09e72bd401a8414c5c0bdaf88bb/bridgewater-associates-engineering-targeted-returns-and-risks-aug-2011.pdf) explains the principles, now widely known as __risk parity__.
> * __Pure Alpha:__ Launched in 1991, Pure Alpha is the firm's actively traded portfolio. It trades on Bridgewater's views of global markets and aims for returns that are uncorrelated with the stock and bond markets: whether Pure Alpha has a good year should not depend on whether those markets went up or down. The name refers to __alpha__, the return earned from skill rather than from holding the markets.
> * __Culture:__ The firm's [culture](https://www.bridgewater.com/culture) is built on what Dalio calls radical truth and radical transparency, described in his book [*Principles: Life and Work*](https://www.principles.com/).

__Explore further:__

* __Jobs and internships:__ Browse the [job openings](https://www.bridgewater.com/working-at-bridgewater/job-openings) and [students](https://www.bridgewater.com/working-at-bridgewater/students) pages. The [2027 Investment Associate Internship Program](https://www.bridgewater.com/2027-investment-associate-internship-program), an eight-week summer program in Westport, Connecticut, is posted as of September 21, 2026.
* __YouTube:__ Watch [How The Economic Machine Works](https://www.youtube.com/watch?v=PHe0bXAIuk0), a thirty-minute animated explanation of credit and debt cycles narrated by Dalio on the [Principles by Ray Dalio](https://www.youtube.com/@principlesbyraydalio) channel, or visit the firm's own [Bridgewater Associates](https://www.youtube.com/@Bridgewater) channel.

__Connection to today's lecture:__ All Weather rests on the observation that a portfolio's risk depends on how its assets move together, not only on each asset's risk. Today we make that idea precise: we estimate a covariance matrix from historical growth rates, extend the price model to reproduce it, and sample long-only weights to see how the allocation changes portfolio growth and risk. Let's begin by extending the single asset model to several assets.

___

## Multiple Asset Geometric Brownian Motion (MAGBM) Model
Simulating $M$ single asset geometric Brownian motion (GBM) models independently assumes zero relationship between the assets' growth rates. To describe firms that respond together to market news, or perhaps respond in opposite directions, we need to capture the relationships between their price movements. We do this by introducing a covariance matrix that describes how the assets' growth rates move together.

Consider a portfolio $\mathcal{P}=\{1,2,\ldots,M\}$ of $M\geq2$ assets, with share prices $S_i(t)>0$ and initial prices $S_i(0)>0$. Here, we index the assets by $i,j\in\mathcal{P}$. Like single asset models, each asset has its own drift parameter $\mu_i$ (units: inverse years). However, the difference between single and multiple asset models lies in the volatility. 

> __Covariance rate matrix__ 
> 
> For multiple assets, we introduce a __covariance rate matrix__ $\mathbf{C}\in\mathbb{R}^{M\times M}$ (units: inverse years) that captures the individual volatilities of each asset, as well as the relationships between assets. The diagonal entries of the covariance rate matrix are the squared volatilities, $C_{ii}=\sigma_i^2$, while the off-diagonal entries describe the __covariance__, i.e., the relationship between two assets' growth rates per unit time. We assume the drifts and covariance rate remain constant over the modeled period.
>
> The covariance rate matrix is symmetric and positive semidefinite, so every weighted combination of the assets has a nonnegative variance rate.

Let $W_1(t),\ldots,W_M(t)$ be independent standard Wiener processes, indexed by $\ell\in\{1,2,\ldots,M\}$. We combine their increments using a __loading matrix__ $\mathbf{A}\in\mathbb{R}^{M\times M}$ whose entries have units of inverse square-root years. We choose the loading matrix $\mathbf{A}$ to reproduce the covariance rate through the identity:
$$
\mathbf{A}\mathbf{A}^{\top}=\mathbf{C}.
$$
The multiple asset GBM model for the share price of asset $i$ is then given by:
$$
\frac{dS_i(t)}{S_i(t)}
=\mu_i\,dt+\underbrace{\sum_{\ell=1}^{M}A_{i\ell}\,dW_\ell(t)}_{\text{correlated noise}},
\qquad i\in\mathcal{P}.
$$
The $i$-th row of $\mathbf{A}$ gives the weights for the $M$ independent Wiener increments that drive asset $i$'s price fluctuations.

> __What does the loading matrix do?__
>
> * __Independent noise becomes correlated noise:__ Every asset uses the same vector of Wiener increments, combined with its own row of $\mathbf{A}$. These shared random inputs make the assets' fluctuations correlated: a large common increment moves every asset with a nonzero loading on it at the same time.
>
> * __The factor reproduces the covariance rate:__ Independent Wiener increments have variance $dt$ and zero covariance with one another. The covariance of the noise terms for assets $i$ and $j$ is therefore given by:
> $$
> \begin{aligned}
> \operatorname{Cov}\!\left(\sum_{\ell=1}^{M}A_{i\ell}\,dW_\ell,
> \sum_{\ell=1}^{M}A_{j\ell}\,dW_\ell\right)
> &=\sum_{\ell=1}^{M}A_{i\ell}A_{j\ell}\,dt\\
> &=(\mathbf{A}\mathbf{A}^{\top})_{ij}\,dt=C_{ij}\,dt.
> \end{aligned}
> $$

For a positive definite $\mathbf{C}$, a __Cholesky decomposition__ gives a lower-triangular factor $\mathbf{A}$. The symmetric positive semidefinite square root $\mathbf{C}^{1/2}$ is another valid choice, including when $\mathbf{C}$ is singular. Glasserman develops this construction and its exact simulation in Section 3.2.3 of [Monte Carlo Methods in Financial Engineering](https://link.springer.com/book/10.1007/978-0-387-21617-1). We will examine these covariance-matrix properties in the next section.

### Exact one-step transition
Applying Itô's lemma to $\ln S_i(t)$ gives each asset its own half-variance correction. The [derivation notebook](advanced/ito-derivation/CHEME-5660-L5b-Derivation-MAGBM-Solution-Fall-2026.ipynb) extends the single asset argument from L4b to several noise sources; here we record the one new step. The correction is half the quadratic variation of the noise term per unit time. Because the Wiener processes are independent, $dW_\ell\,dW_m=0$ for $\ell\neq m$ and $dW_\ell^2=dt$, so:
$$
\Bigl(\sum_{\ell=1}^{M}A_{i\ell}\,dW_\ell\Bigr)^2
=\sum_{\ell=1}^{M}\sum_{m=1}^{M}A_{i\ell}A_{im}\,dW_\ell\,dW_m
=\sum_{\ell=1}^{M}A_{i\ell}^{2}\,dt.
$$
The row sum of squares is $\sum_{\ell=1}^{M}A_{i\ell}^{2}=(\mathbf{A}\mathbf{A}^{\top})_{ii}=C_{ii}$, the $i=j$ case of the covariance calculation above. Following L4b's notation, the __mean growth rate__ of asset $i$ is given by:
$$
\mu_{g,i}
=\mu_i-\frac{1}{2}\sum_{\ell=1}^{M}A_{i\ell}^{2}
=\mu_i-\frac{C_{ii}}{2}.
$$
Because $C_{ii}=\sigma_i^2$, this is the same correction used in the single asset model, and $\mu_i=\mu_{g,i}+C_{ii}/2$.

As in L4b, define the time grid $t_k=k\Delta t$ for $k=0,1,\ldots,N$, where $\Delta t>0$ is measured in years and $T=N\Delta t$. Let $\mathbf{Z}_k\sim\mathcal{N}(\mathbf{0},\mathbf{I}_M)$ be a vector of $M$ independent standard normal random variables (shocks) drawn at step $k$. The exact one-step transition is given by:
$$
\boxed{
S_i(t_k)
=S_i(t_{k-1})\exp\!\left[\mu_{g,i}\Delta t
+\sqrt{\Delta t}\,(\mathbf{A}\mathbf{Z}_k)_i\right],
\qquad i\in\mathcal{P},\quad k=1,2,\ldots,N.
}
$$
Here, $(\mathbf{A}\mathbf{Z}_k)_i=\sum_{\ell=1}^{M}A_{i\ell}Z_{k,\ell}$ is the noise contribution for asset $i$ before scaling by $\sqrt{\Delta t}$. We use the __same vector $\mathbf{Z}_k$ for all assets within each time step__ and draw a __new independent vector at every step__, so the shocks $\mathbf{Z}_1,\ldots,\mathbf{Z}_N$ are independent. This preserves the covariance between assets and the independent increments through time.

For constant parameters, the transition gives exact model prices at the grid points. As in the single asset case, prices at a single fixed horizon $T$ can also be sampled directly from the initial prices using one vector $\mathbf{Z}$ scaled by $\sqrt{T}$.

To use the model, we need to estimate $\mathbf{C}$ from data. Let's now construct the empirical growth-rate covariance matrix and use a scaling rule to obtain the covariance rate.

___

## Empirical Covariance Matrix
A covariance matrix records the variance of each feature and the pairwise covariance between features. Here, the features are the growth rates of the firms in $\mathcal{P}$, and each sample is an aligned vector of their growth rates over one period.

Let's consider samples from two-dimensional normal distributions with negative, zero, and positive covariance. The ellipses are contours of constant probability density, drawn at one and two standard deviations along the principal axes of each distribution, and their tilt shows the sign of the covariance. 


<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L5b-Covariance-Schematic.svg" width="880" alt="Three columns show samples with negative, zero, and positive covariance. In the top row each blue cloud carries a solid one standard deviation ellipse and a dashed two standard deviation ellipse whose tilt follows the sign of the covariance. The bottom row overlays each blue cloud with a red cloud whose covariance matrix is four times larger and draws the dashed two standard deviation ellipse of each cloud; the red ellipse is the blue cloud's ellipse scaled by two, so the standard deviations double while the correlation is preserved."/>
    </center>
</div>


In the lower row, each blue cloud is compared with a red cloud whose covariance matrix is four times larger. This doubles the standard deviations while preserving the correlation, so the dashed two standard deviation contour of the red cloud is the contour of the blue cloud scaled by two.

We build the estimate in three steps: define the pairwise sample covariance and correlation, compute all pairs at once from a data matrix, and convert the result to the covariance rate $\mathbf{C}$ that the model needs.

### Covariance and correlation
Suppose we have $N\geq2$ equally spaced (aligned) time periods of growth-rate data for the $M$ firms in $\mathcal{P}$.  Let $g_k^{(i)}$ be the growth rate of firm $i$ in period $k=1,2,\ldots,N$. We use $i$ and $j$ to index firms and $k$ to index time periods.

Collect firm $i$'s observations into the vector $\mathbf{g}^{(i)}=[g_1^{(i)},\ldots,g_N^{(i)}]^{\top}$ with the mean:
$$
g'_i=\frac{1}{N}\sum_{k=1}^{N}g_k^{(i)}.
$$
The empirical growth-rate covariance matrix $\hat{\mathbf{\Sigma}}_g\in\mathbb{R}^{M\times M}$ collects the pairwise sample covariances:
$$
\hat{\Sigma}_{g,ij}
=\frac{1}{N-1}\underbrace{\sum_{k=1}^{N}
\overbrace{\bigl(g_k^{(i)}-g'_i\bigr)}^{\text{deviation from mean}}
\bigl(g_k^{(j)}-g'_j\bigr)}_{\text{sum over time periods of product of deviations}},
\qquad i,j\in\mathcal{P}.
$$
When $i=j$, the two deviations are the same, so the diagonal entry $\hat{\Sigma}_{g,ii}$ is the sample variance of firm $i$'s growth rate. Its square root is the growth-rate standard deviation $\sigma_g$ from L3a (units: inverse years). An off-diagonal entry describes how two firms' growth rates vary together. 

When both sample variances are positive, we obtain the dimensionless __correlation__ between firms $i$ and $j$ by dividing the covariance by the product of the two standard deviations:
$$
\boxed{
\rho_{ij}
=\frac{\hat{\Sigma}_{g,ij}}{\sqrt{\hat{\Sigma}_{g,ii}\,\hat{\Sigma}_{g,jj}}}
\in[-1,1]
\quad\Longleftrightarrow\quad
\hat{\Sigma}_{g,ij}
=\rho_{ij}\sqrt{\hat{\Sigma}_{g,ii}\,\hat{\Sigma}_{g,jj}}.
}
$$
__Correlation__ compares the strength of linear relationships across pairs of firms. Zero correlation means no linear relationship in this sample; it does not imply that the firms' growth rates are independent. The covariance retains the growth-rate scales needed by the model.

### The data matrix and the sample covariance
To compute all pairwise covariances at once, arrange the growth rates in a __data matrix__ $\mathbf{G}\in\mathbb{R}^{N\times M}$. Rows are time periods and columns are firms, so row $k$ contains the growth rates of all $M$ firms over the same interval:
$$
\mathbf{G}=\begin{bmatrix}
g_1^{(1)} & g_1^{(2)} & \cdots & g_1^{(M)} \\
g_2^{(1)} & g_2^{(2)} & \cdots & g_2^{(M)} \\
\vdots & \vdots & \ddots & \vdots \\
g_N^{(1)} & g_N^{(2)} & \cdots & g_N^{(M)}
\end{bmatrix}.
$$
To center the data, subtract each firm's sample mean from its column. Let $\mathbf{g}^{\prime}=[g'_1,g'_2,\ldots,g'_M]^{\top}$ be the vector of sample means. The centered data matrix is given by:
$$
\tilde{\mathbf{G}}=\mathbf{G}-\mathbf{1}\,\mathbf{g}^{\prime\top},
$$
where $\mathbf{1}\in\mathbb{R}^{N}$ is a vector of ones. The product $\mathbf{1}\,\mathbf{g}^{\prime\top}$ is an $N\times M$ matrix with the sample means in every row. Column $i$ of $\tilde{\mathbf{G}}$ is the centered observation vector $\tilde{\mathbf{g}}^{(i)}=\mathbf{g}^{(i)}-g'_i\mathbf{1}$.

> __Outer product:__ The matrix $\mathbf{1}\,\mathbf{g}^{\prime\top}$ is an example of an outer product. The [outer product](https://en.wikipedia.org/wiki/Outer_product) of two vectors $\mathbf{a}\in\mathbb{R}^{N}$ and $\mathbf{b}\in\mathbb{R}^{M}$ is the $N\times M$ matrix $\mathbf{a}\mathbf{b}^{\top}$ with elements $(\mathbf{a}\mathbf{b}^{\top})_{kj}=a_kb_j$.

The empirical growth-rate covariance matrix can now be computed with one matrix product, whose entries are dot products of the centered columns:
$$
\boxed{
\hat{\mathbf{\Sigma}}_g
=\frac{1}{N-1}\tilde{\mathbf{G}}^{\top}\tilde{\mathbf{G}}
\quad\Longleftrightarrow\quad
\hat{\Sigma}_{g,ij}
=\frac{1}{N-1}\,\tilde{\mathbf{g}}^{(i)\top}\tilde{\mathbf{g}}^{(j)}
=\frac{1}{N-1}\sum_{k=1}^{N}\tilde{g}_k^{(i)}\tilde{g}_k^{(j)}.
}
$$
The $(i,j)$ entry of $\tilde{\mathbf{G}}^{\top}\tilde{\mathbf{G}}$ is the dot product of the centered columns for firms $i$ and $j$, and each term $\tilde{g}_k^{(i)}=g_k^{(i)}-g'_i$ is the deviation from the mean in period $k$. Dividing by $N-1$ therefore gives the same pairwise covariance defined above, for every pair of firms at once.

The centered-data formula also tells us what to expect from the estimate:

> __Covariance Matrix Properties:__
>
> * __Elements:__ The diagonal entries $\hat{\Sigma}_{g,ii}$ are the sample growth-rate variances, so they are non-negative. The off-diagonal entries $\hat{\Sigma}_{g,ij}$ are the sample covariances between firms $i$ and $j$.
> * __Symmetry:__ Swapping $i$ and $j$ leaves the products in the pairwise covariance formula unchanged, so $\hat{\Sigma}_{g,ij}=\hat{\Sigma}_{g,ji}$.
> * __Positive semidefinite:__ For any $\mathbf{v}\in\mathbb{R}^{M}$,
> $$
> \mathbf{v}^{\top}\hat{\mathbf{\Sigma}}_g\mathbf{v}
> =\frac{1}{N-1}\mathbf{v}^{\top}\tilde{\mathbf{G}}^{\top}\tilde{\mathbf{G}}\mathbf{v}
> =\frac{1}{N-1}\lVert\tilde{\mathbf{G}}\mathbf{v}\rVert_2^2
> \geq0,
> $$
> so the sample variance of any weighted sum of the growth rates is non-negative.

Finally, the price model needs the covariance rate $\mathbf{C}$. Because the model's growth rates over one step have covariance $\mathbf{C}/\Delta t$ (see the [derivation notebook](advanced/ito-derivation/CHEME-5660-L5b-Derivation-MAGBM-Solution-Fall-2026.ipynb)), we scale the sample covariance by the time step to obtain the estimate:
$$
\boxed{
\hat{\mathbf{C}}=\Delta t\,\hat{\mathbf{\Sigma}}_g.
}
$$
Let's compute the empirical covariance matrix for the firms in our dataset and examine the relationships it describes.

> __Example:__
>
> [▶ Compute the covariance matrix for our dataset](CHEME-5660-L5b-Example-CovarianceMatrix-Fall-2026.ipynb). We compute the empirical growth-rate covariance matrix and convert it to the GBM covariance rate. We check the covariance calculation, compare the corresponding volatilities with our L4b estimates, and interpret the covariance and correlation of a pair of firms.

The estimated covariance tells us how asset growth rates vary together. Next, let's describe how much of our investment to assign to each asset.

___


## Portfolio Weights and Dirichlet Sampling
Once we can simulate several assets together, the next question is how much of our money to put into each asset. Let total initial wealth we want to invest  be $W_0>0$ dollars, and let $\mathbf{w}=(w_1,\ldots,w_M)^{\top}$ contain the fractions invested in assets $i, i\in\mathcal{P}$, at time zero. For a __fully invested, long-only__ portfolio, the weights satisfy:
$$
w_i\geq0,\qquad \sum_{i\in\mathcal{P}}w_i=1.
$$
These constraints define the $(M-1)$-dimensional __simplex__ $\Delta^{M-1}$: a line segment for two assets, a triangle for three, and so on. The amount invested in asset $i$ is $w_iW_0$, so we buy $n_i=w_iW_0/S_i(0)$ shares. We allow fractional shares but ignore dividends and trading costs. Holding these share counts fixed, the value of our holding in asset $i$ at time $t$ is $n_iS_i(t)$. The __buy-and-hold__ portfolio wealth is the sum of these holding values. Substituting the share counts gives:
$$
\begin{aligned}
W_t &= \sum_{i\in\mathcal{P}}n_iS_i(t)
    && \text{sum the holding values} \\
    &= \sum_{i\in\mathcal{P}}\left(\frac{w_iW_0}{S_i(0)}\right)S_i(t)
    && \text{substitute the share counts} \\
    &= W_0\sum_{i\in\mathcal{P}}w_i\frac{S_i(t)}{S_i(0)}.
    && \text{factor out } W_0
\end{aligned}
$$
Dividing both sides by the initial budget gives wealth relative to $W_0$:
$$
\boxed{
\frac{W_t}{W_0}=\sum_{i\in\mathcal{P}}w_i\frac{S_i(t)}{S_i(0)}.
}
$$
Each ratio $S_i(t)/S_i(0)$ measures an asset's price relative to its initial price. Thus, wealth relative to the initial budget is a weighted average of these price ratios using the __initial__ portfolio weights. The share counts remain fixed, but the fraction of wealth in each asset changes as relative prices move. At time $t$, that fraction is given by:
$$
w_i(t)=\frac{n_iS_i(t)}{W_t}.
$$
Thus, the $\mathbf{w}$ vector specifies the initial allocation. Interestingly, maintaining _constant weights_ requires trading to rebalance the portfolio, with its own turnover and cost assumptions. 

### Sampling candidate allocations
__How should we choose the portfolio weights?__ Later in the course, we will solve for optimal weights. Today, we take a simpler approach: draw many random allocations from the simplex and compare the portfolios they produce. 

The Dirichlet distribution is well suited to this because every draw is a valid long-only allocation.

> __Dirichlet portfolio weights__
>
> Let $\boldsymbol{\alpha}=(\alpha_1,\ldots,\alpha_M)$ contain positive __concentration parameters__, and let $\alpha_0=\sum_{i\in\mathcal{P}}\alpha_i$. A random weight vector $\mathbf{w}\sim\operatorname{Dirichlet}(\boldsymbol{\alpha})$ always has components that sum to one, and a component equal to zero has probability zero, so every weight is positive in practice. Each draw is an initial allocation. The component moments are given by:
> $$
> \begin{aligned}
> \mathbb{E}[w_i]&=\frac{\alpha_i}{\alpha_0},\\
> \operatorname{Var}(w_i)&=\frac{\alpha_i(\alpha_0-\alpha_i)}{\alpha_0^2(\alpha_0+1)},\\
> \operatorname{Cov}(w_i,w_j)&=-\frac{\alpha_i\alpha_j}{\alpha_0^2(\alpha_0+1)},\qquad i\ne j.
> \end{aligned}
> $$
> The covariances are negative because the weights divide a fixed budget: a larger allocation to one asset leaves less for the others.

With symmetric concentrations $\alpha_i=\alpha$, the value $\alpha=1$ spreads the draws evenly over the simplex, values below one push them toward the boundary where a few assets hold most of the budget, and large values cluster them near equal weights $1/M$. Unequal concentrations tilt the expected allocation toward assets with larger $\alpha_i$, which lets us explore different allocations before evaluating their growth and risk.

### Comparing growth and risk
How do we compare the portfolios that different allocations produce? We summarize the performance of each allocation by the mean and variance of its growth rate.

> __Proposition: Linear Portfolio Growth-Rate Proxy__
>
> Hold an allocation $\mathbf{w}$ for one period of length $\Delta t$, and collect the asset growth rates over that period in $\mathbf{g}=[g_1,\ldots,g_M]^{\top}$. Keeping only the terms that are first order in the log returns $g_i\Delta t$, the portfolio growth rate from the buy-and-hold wealth formula is approximated by the __linear growth-rate proxy__ $g_p$:
>
> $$
> \frac{1}{\Delta t}\ln\!\left(\frac{W_{\Delta t}}{W_0}\right)
> =\frac{1}{\Delta t}\ln\!\left(\sum_{i\in\mathcal{P}}w_i e^{g_i\Delta t}\right)
> \approx\mathbf{w}^{\top}\mathbf{g}=g_p.
> $$
>
> Applying the proxy to each historical growth-rate observation, its sample mean and sample variance are given by:
>
> $$
> \begin{aligned}
> g^{\prime}_p&=\mathbf{w}^{\top}\mathbf{g}^{\prime}, &&\text{(mean growth rate)}\\
> \sigma_{g,p}^2&=\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_g\mathbf{w}, &&\text{(variance)}
> \end{aligned}
> $$
>
> where $\mathbf{g}^{\prime}$ ($\mathrm{year}^{-1}$) and $\hat{\mathbf{\Sigma}}_g$ ($\mathrm{year}^{-2}$) are the sample mean vector and covariance matrix from the empirical covariance section. The proxy describes a single period. The exact portfolio log growth differs from $\mathbf{w}^{\top}\mathbf{g}$, and buy-and-hold weights drift over multiple periods, so we use the wealth formula to follow a portfolio through time.
>
> __Where is this coming from?__ Using $e^{g_i\Delta t}\approx1+g_i\Delta t$, then $\sum_{i}w_i=1$, and then $\ln(1+x)\approx x$ gives $\ln\sum_{i}w_ie^{g_i\Delta t}\approx\Delta t\,\mathbf{w}^{\top}\mathbf{g}$.

Sampling gives us candidate allocations to compare using these measures. The best sampled candidate need not be optimal over all feasible weights; finding an optimum is the topic of L6a. Let's examine how the concentration parameters change the candidates and the portfolios they produce.

> __Example:__
>
> [▶ Sample portfolio weights with the Dirichlet distribution](CHEME-5660-L5b-Example-Dirichlet-PortfolioWeights-Fall-2026.ipynb). We draw long-only allocations, examine how the concentration parameters shape the weights, and compare the estimated growth and risk of the sampled portfolios. We also follow selected buy-and-hold portfolios to see how their wealth and weights change as prices move.

This comparison prepares us to choose weights using an explicit growth and risk objective in L6a.

___

## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L6a; the [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ Derivation of the multiple asset GBM solution](advanced/ito-derivation/CHEME-5660-L5b-Derivation-MAGBM-Solution-Fall-2026.ipynb). Why does each asset's drift correction contain only its own variance rate $C_{ii}$ and none of the covariances? We state the multiplication rules for the increments of independent Wiener processes, extend Itô's lemma from L4b to several noise sources, and apply it to the log price of each asset. Integrating on the time grid recovers the boxed one-step transition, and writing the growth rates in vector form gives $\operatorname{Cov}(\mathbf{g})=\mathbf{C}/\Delta t$, the scaling relation used to estimate the covariance rate from data.

* [▶ Sampling error and shrinkage in covariance estimation](advanced/covariance-estimation/CHEME-5660-L5b-Advanced-CovarianceEstimation-Fall-2026.ipynb). How does uncertainty in a covariance estimate affect portfolio selection? We compare the sample correlation eigenvalues with a reference for independent data, the Marchenko–Pastur law, and measure estimation error using simulated samples. We then introduce shrinkage, which combines the sample covariance with a simpler target matrix, and compare the resulting minimum-variance portfolios on a later data window.

* [▶ Rolling correlations](advanced/rolling-correlation/CHEME-5660-L5b-Advanced-RollingCorrelation-Fall-2026.ipynb). Are correlations between firms stable over time? We estimate correlations from 2014 to 2024 using rolling windows and exponentially weighted observations, then compare them with the full-sample estimates. We examine how average correlation changes during periods of high market volatility and what these changes mean for a model with a constant covariance rate.

___



## Summary
In this lecture, we extended geometric Brownian motion from one asset to many correlated assets, used historical growth rates to estimate their covariance, and introduced Dirichlet sampling as a way to explore portfolio weights on the long-only simplex.

> __Key Takeaways:__
>
> * **Correlated asset prices:** We reviewed how exponential weights update mean growth and volatility, with the half-life setting how quickly the estimates respond. We then extended single asset geometric Brownian motion by using a loading matrix to combine independent Wiener increments. This gave us a model with a specified covariance rate, a mean growth rate for each asset, and an exact transition for generating correlated price paths.
>
> * **Covariance from data:** We constructed the empirical covariance matrix from centered growth rates and multiplied it by the time step to obtain the model's covariance rate. This connects historical growth-rate variation to asset volatilities and co-movement. Correlation rescales each covariance by the two standard deviations, so we can compare the strength of co-movement across pairs of firms.
>
> * **Portfolio weights:** We used the Dirichlet distribution to explore long-only allocations on the simplex and compared candidates using the estimated mean and variance of a linear growth-rate proxy. We also expressed buy-and-hold wealth in terms of fixed share counts, which explains why investment weights change as prices move.

Next time, we turn the estimated means and covariance into the minimum-variance portfolio and the efficient frontier.

___



## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.
